In [14]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [15]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

In [16]:
df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

In [17]:
df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,9775.522707,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,10725.051692,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,8629.945617,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,8447.759415,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9281.534430,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10374.147811,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,10909.112726,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11430.790829,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12007.048703,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12502.021651,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [18]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['VABI_1'] = lv['VABI'].shift(1)
lv['YUA_2'] = lv['YUA'].shift(2)

In [19]:
lv = lv.dropna()

In [20]:
sm = StructureModel()

In [21]:
sm.add_edges_from([
    ('EUPC', 'QOA'),
    ('VABI', 'QOA'),
    ('EUPC', 'REST'),
    ('EUPC', 'EBSG'),
    ('VABI_1', 'EBSG'),
    ('EBSG', 'LBGD'),
    ('EBSG', 'FDI'),
    ('VABI', 'FDI'),
    ('EBSG', 'PC'),
    ('LBGD', 'ARP'),
    ('YUA_2', 'EUPC')
])

In [22]:
sm.edges

OutEdgeView([('EUPC', 'QOA'), ('EUPC', 'REST'), ('EUPC', 'EBSG'), ('VABI', 'QOA'), ('VABI', 'FDI'), ('EBSG', 'LBGD'), ('EBSG', 'FDI'), ('EBSG', 'PC'), ('VABI_1', 'EBSG'), ('LBGD', 'ARP'), ('YUA_2', 'EUPC')])

In [23]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lv_1.html")

Graphs/fully_connected_lv_1.html


In [24]:
bn = BayesianNetwork(sm)

In [27]:
discretised_lv = pd.DataFrame(index=lv.index)

for col in lv.columns:
    no_unique = lv[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lv)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
2      0     2    0    2    2    2     0     1     2     0    0   0    2   
3      0     2    0    2    0    2     0     1     1     0    0   0    0   
4      0     2    0    0    0    2     0     0     1     1    2   0    0   
5      0     2    0    0    1    2     0     0     0     1    2   0    0   
6      0     1    0    2    0    2     0     0     0     2    1   0    0   
7      1     0    1    0    0    1     1     1     0     1    2   2    0   
8      1     0    1    1    2    1     1     1     1     1    1   1    1   
9      1     0    1    0    2    1     1     2     0     2    0   1    1   
10     1     0    1    0    0    1     1     2     1     2    2   1    1   
11     2     0    2    2    2    0     2     2     0     2    0   2    2   
12     2     0    2    2    2    0     2     2     2     2    1   2    2   
13     2     1    2    1    1    0     2     2     2     0    0   2   

In [28]:
discretised_lv = discretised_lv.reset_index(drop=True)
bn.fit_node_states(discretised_lv)
baseline_auc = utils.get_avg_auc_all_info(discretised_lv, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 6.12190580368042 seconds
Processing fold 1 using 7 cores takes 5.725476026535034 seconds
Processing fold 2 using 7 cores takes 5.842650890350342 seconds
Processing fold 3 using 7 cores takes 5.847394943237305 seconds
Processing fold 4 using 7 cores takes 5.566529989242554 seconds
Baseline AUC: 0.5646464646464647


In [32]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'QOA')]
edges_to_remove = [('EUPC', 'QOA')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [33]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lv_2.html")

Graphs/node_added_lv_2.html


In [34]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'QOA': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.645888090133667 seconds
Processing fold 1 using 7 cores takes 5.983020067214966 seconds
Processing fold 2 using 7 cores takes 6.149681329727173 seconds
Processing fold 3 using 7 cores takes 6.875366926193237 seconds
Processing fold 4 using 7 cores takes 6.59821891784668 seconds
AUC from adding LV between 'EUPC' and 'QOA': 0.561489898989899


In [35]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'REST')]
edges_to_remove = [('EUPC', 'REST')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [36]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lv_3.html")

Graphs/node_added_lv_3.html


In [37]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'REST': {proposed_auc}")

Processing fold 0 using 7 cores takes 7.486265182495117 seconds
Processing fold 1 using 7 cores takes 6.195768117904663 seconds
Processing fold 2 using 7 cores takes 5.94785475730896 seconds
Processing fold 3 using 7 cores takes 5.922776937484741 seconds
Processing fold 4 using 7 cores takes 5.79246187210083 seconds
AUC from adding LV between 'EUPC' and 'REST': 0.6007575757575757


In [38]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'EBSG')]
edges_to_remove = [('EUPC', 'EBSG')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [39]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lv_4.html")

Graphs/node_added_lv_4.html


In [40]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'EBSG': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.634697914123535 seconds
Processing fold 1 using 7 cores takes 5.923002004623413 seconds
Processing fold 2 using 7 cores takes 6.0919349193573 seconds
Processing fold 3 using 7 cores takes 5.918480157852173 seconds
Processing fold 4 using 7 cores takes 5.952781915664673 seconds
AUC from adding LV between 'EUPC' and 'EBSG': 0.5954545454545455


In [41]:
edges_to_add = [('LV', 'YUA_2'), ('LV', 'EUPC')]
edges_to_remove = [('YUA_2', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [42]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lv_5.html")

Graphs/node_added_lv_5.html


In [43]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'YUA_2' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.773053169250488 seconds
Processing fold 1 using 7 cores takes 5.7691261768341064 seconds
Processing fold 2 using 7 cores takes 5.817845106124878 seconds
Processing fold 3 using 7 cores takes 5.562762022018433 seconds
Processing fold 4 using 7 cores takes 5.44517707824707 seconds
AUC from adding LV between 'YUA_2' and 'EUPC': 0.7257575757575758


In [44]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [45]:
df_lv_1 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

In [46]:
df_lv_1 = df_lv_1.drop(["Unnamed: 0"], axis = 1)

df_lv_1

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,9775.522707,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,10725.051692,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,8629.945617,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,8447.759415,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9281.534430,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10374.147811,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,10909.112726,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11430.790829,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12007.048703,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12502.021651,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [47]:
lv_1 = pd.DataFrame()
lv_1 = df_lv_1.copy()
lv_1['EUPC_1'] = lv_1['EUPC'].shift(1)
lv_1['CO2C_2'] = lv_1['CO2C'].shift(2)

In [48]:
lv_1 = lv_1.dropna()

In [49]:
sm1 = StructureModel()

In [50]:
sm1.add_edges_from([
    ('EUPC_1', 'QOR'),  
     ('CO2C_2', 'QOR')
])

In [51]:
sm1.edges

OutEdgeView([('EUPC_1', 'QOR'), ('CO2C_2', 'QOR')])

In [52]:
viz = plot_structure(
    sm1,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lv_3.html")

Graphs/fully_connected_lv_3.html


In [53]:
bn1 = BayesianNetwork(sm1)

In [54]:
discretised_lv = pd.DataFrame(index=lv_1.index)

for col in lv_1.columns:
    no_unique = lv_1[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv_1[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv_1[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lv)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
2      0     2    0    2    2    2     0     1     2     0    0   0    2   
3      0     2    0    2    0    2     0     1     1     0    0   0    0   
4      0     2    0    0    0    2     0     0     1     1    2   0    0   
5      0     2    0    0    1    2     0     0     0     1    2   0    0   
6      0     1    0    2    0    2     0     0     0     2    1   0    0   
7      1     0    1    0    0    1     1     1     0     1    2   2    0   
8      1     0    1    1    2    1     1     1     1     1    1   1    1   
9      1     0    1    0    2    1     1     2     0     2    0   1    1   
10     1     0    1    0    0    1     1     2     1     2    2   1    1   
11     2     0    2    2    2    0     2     2     0     2    0   2    2   
12     2     0    2    2    2    0     2     2     2     2    1   2    2   
13     2     1    2    1    1    0     2     2     2     0    0   2   

In [55]:
discretised_lv = discretised_lv.reset_index(drop=True)
bn1.fit_node_states(discretised_lv)
baseline_auc = utils.get_avg_auc_all_info(discretised_lv, bn1)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 4.594862937927246 seconds
Processing fold 1 using 7 cores takes 4.669683933258057 seconds
Processing fold 2 using 7 cores takes 4.680412769317627 seconds
Processing fold 3 using 7 cores takes 4.809460163116455 seconds
Processing fold 4 using 7 cores takes 4.778480052947998 seconds
Baseline AUC: 0.5166666666666666


In [56]:
edges_to_add = [('LV', 'EUPC_1'), ('LV', 'QOR')]
edges_to_remove = [('EUPC_1', 'QOR')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [57]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_pl_3.html")

Graphs/node_added_pl_3.html


In [58]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_1' and 'QOR': {proposed_auc}")

Processing fold 0 using 7 cores takes 5.257943630218506 seconds
Processing fold 1 using 7 cores takes 4.7214930057525635 seconds
Processing fold 2 using 7 cores takes 4.821254014968872 seconds
Processing fold 3 using 7 cores takes 4.999223947525024 seconds
Processing fold 4 using 7 cores takes 4.725778102874756 seconds
AUC from adding LV between 'EUPC_1' and 'QOR': 0.5217592592592591


In [108]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [109]:
df_lv_1 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

In [110]:
df_lv_1 = df_lv_1.drop(["Unnamed: 0"], axis = 1)

df_lv_1

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,9775.522707,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,10725.051692,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,8629.945617,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,8447.759415,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9281.534430,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10374.147811,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,10909.112726,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11430.790829,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12007.048703,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12502.021651,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [111]:
lv_1 = pd.DataFrame()
lv_1 = df_lv_1.copy()
lv_1['EUPC_2'] = lv_1['EUPC'].shift(2)

In [112]:
lv_1 = lv_1.dropna()

In [113]:
sm1 = StructureModel()

In [114]:
sm1.add_edges_from([
    ('EUPC_2', 'FDI'),  
     ('EBSG', 'FDI'),
    ('VABI', 'FDI'),
    ('EBSG', 'PC'),
    ('VABI', 'QOA'),
    ('EUPC_2', 'CO2C')
])

In [115]:
sm1.edges

OutEdgeView([('EUPC_2', 'FDI'), ('EUPC_2', 'CO2C'), ('EBSG', 'FDI'), ('EBSG', 'PC'), ('VABI', 'FDI'), ('VABI', 'QOA')])

In [116]:
viz = plot_structure(
    sm1,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lv_4.html")

Graphs/fully_connected_lv_4.html


In [117]:
bn1 = BayesianNetwork(sm1)

In [118]:
discretised_lv = pd.DataFrame(index=lv_1.index)

for col in lv_1.columns:
    no_unique = lv_1[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv_1[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv_1[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lv)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
2      0     1    0    1    1    1     0     1     1     0    0   0    1   
3      0     1    0    1    0    1     0     0     1     0    0   0    0   
4      0     1    0    0    0    1     0     0     1     0    1   0    0   
5      0     1    0    0    0    1     0     0     0     1    1   0    0   
6      0     1    0    1    0    1     0     0     0     1    1   0    0   
7      0     0    0    0    0    1     1     0     0     1    1   1    0   
8      0     0    0    0    1    0     0     1     0     0    0   1    1   
9      1     0    1    0    1    1     0     1     0     1    0   0    0   
10     1     0    1    0    0    0     1     1     0     1    1   1    1   
11     1     0    1    1    1    0     1     1     0     1    0   1    1   
12     1     0    1    1    1    0     1     1     1     1    1   1    1   
13     1     0    1    0    0    0     1     1     1     0    0   1   

In [119]:
discretised_lv = discretised_lv.reset_index(drop=True)
bn1.fit_node_states(discretised_lv)
baseline_auc = utils.get_avg_auc_all_info(discretised_lv, bn1)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 6.374677419662476 seconds
Processing fold 1 using 7 cores takes 5.834593057632446 seconds
Processing fold 2 using 7 cores takes 5.847355127334595 seconds
Processing fold 3 using 7 cores takes 5.785099744796753 seconds
Processing fold 4 using 7 cores takes 5.950894117355347 seconds
Baseline AUC: 0.6063492063492063


In [120]:
edges_to_add = [('LV', 'EUPC_2'), ('LV', 'FDI')]
edges_to_remove = [('EUPC_2', 'FDI')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [121]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lv_4.html")

Graphs/node_added_lv_4.html


In [122]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_2' and 'FDI': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.440664052963257 seconds
Processing fold 1 using 7 cores takes 5.741899013519287 seconds
Processing fold 2 using 7 cores takes 5.961413145065308 seconds
Processing fold 3 using 7 cores takes 5.880470037460327 seconds
Processing fold 4 using 7 cores takes 5.863632917404175 seconds
AUC from adding LV between 'EUPC_2' and 'FDI': 0.6031746031746031


In [ ]:
edges_to_add = [('LV', 'EUPC_2'), ('LV', 'CO2C')]
edges_to_remove = [('EUPC_2', 'CO2C')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [123]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lv_5.html")

Graphs/node_added_lv_5.html


In [124]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_2' and 'C02C': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.283869981765747 seconds
Processing fold 1 using 7 cores takes 6.015789270401001 seconds
Processing fold 2 using 7 cores takes 5.667325973510742 seconds
Processing fold 3 using 7 cores takes 5.746006011962891 seconds
Processing fold 4 using 7 cores takes 5.712677240371704 seconds
AUC from adding LV between 'EUPC_2' and 'C02C': 0.6031746031746031


In [77]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [78]:
df_lv_1 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

In [79]:
df_lv_1 = df_lv_1.drop(["Unnamed: 0"], axis = 1)

df_lv_1

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,9775.522707,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,10725.051692,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,8629.945617,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,8447.759415,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9281.534430,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10374.147811,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,10909.112726,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11430.790829,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12007.048703,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12502.021651,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [80]:
lv_1 = pd.DataFrame()
lv_1 = df_lv_1.copy()
lv_1['VABI_1'] = lv_1['VABI'].shift(1)
lv_1['YUA_2'] = lv_1['YUA'].shift(2)

In [81]:
lv_1 = lv_1.dropna()

In [82]:
sm1 = StructureModel()

In [83]:
sm1.add_edges_from([
    ('QOA', 'EUPC'),
    ('VABI', 'QOA'),
    ('EUPC', 'REST'),
    ('EUPC', 'EBSG'),
    ('VABI_1', 'EBSG'),
    ('EBSG', 'LBGD'),
    ('EBSG', 'FDI'),
    ('VABI', 'FDI'),
    ('EBSG', 'PC'),
    ('LBGD', 'ARP'),
    ('YUA_2', 'EUPC')
])

In [84]:
sm1.edges

OutEdgeView([('QOA', 'EUPC'), ('EUPC', 'REST'), ('EUPC', 'EBSG'), ('VABI', 'QOA'), ('VABI', 'FDI'), ('EBSG', 'LBGD'), ('EBSG', 'FDI'), ('EBSG', 'PC'), ('VABI_1', 'EBSG'), ('LBGD', 'ARP'), ('YUA_2', 'EUPC')])

In [85]:
viz = plot_structure(
    sm1,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lv_5.html")

Graphs/fully_connected_lv_5.html


In [86]:
bn1 = BayesianNetwork(sm1)

In [87]:
discretised_lv = pd.DataFrame(index=lv_1.index)

for col in lv_1.columns:
    no_unique = lv_1[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv_1[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv_1[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lv)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
2      0     2    0    2    2    2     0     1     2     0    0   0    2   
3      0     2    0    2    0    2     0     1     1     0    0   0    0   
4      0     2    0    0    0    2     0     0     1     1    2   0    0   
5      0     2    0    0    1    2     0     0     0     1    2   0    0   
6      0     1    0    2    0    2     0     0     0     2    1   0    0   
7      1     0    1    0    0    1     1     1     0     1    2   2    0   
8      1     0    1    1    2    1     1     1     1     1    1   1    1   
9      1     0    1    0    2    1     1     2     0     2    0   1    1   
10     1     0    1    0    0    1     1     2     1     2    2   1    1   
11     2     0    2    2    2    0     2     2     0     2    0   2    2   
12     2     0    2    2    2    0     2     2     2     2    1   2    2   
13     2     1    2    1    1    0     2     2     2     0    0   2   

In [88]:
discretised_lv = discretised_lv.reset_index(drop=True)
bn1.fit_node_states(discretised_lv)
baseline_auc = utils.get_avg_auc_all_info(discretised_lv, bn1)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 6.158768177032471 seconds
Processing fold 1 using 7 cores takes 5.610931158065796 seconds
Processing fold 2 using 7 cores takes 5.697483062744141 seconds
Processing fold 3 using 7 cores takes 5.8710291385650635 seconds
Processing fold 4 using 7 cores takes 5.732547760009766 seconds
Baseline AUC: 0.5992424242424241


In [89]:
edges_to_add = [('LV', 'QOA'), ('LV', 'EUPC')]
edges_to_remove = [('QOA', 'EUPC')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [90]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lv_5.html")

Graphs/node_added_lv_5.html


In [91]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'QOA' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.712707042694092 seconds
Processing fold 1 using 7 cores takes 5.89710807800293 seconds
Processing fold 2 using 7 cores takes 5.935099124908447 seconds
Processing fold 3 using 7 cores takes 5.803646802902222 seconds
Processing fold 4 using 7 cores takes 5.8999388217926025 seconds
AUC from adding LV between 'QOA' and 'EUPC': 0.5267676767676768
